In [1]:
import pandas as pd 
import numpy as np
import requests
from io import StringIO
from datetime import timedelta, datetime
#ignore warnings
import warnings
warnings.filterwarnings('ignore')

/Users/stefania/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
df_parquet =  pd.read_parquet("../../../../input_data/SMH_projections_age65plus.parquet")
df_parquet.model_name.unique()

array(['MOBS_NEU-GLEAM_FLU', 'NotreDame-FRED', 'USC-SIkJalpha',
       'UVA-FluXSim', 'UVA-EscapeFlu'], dtype=object)

In [25]:
def read_csv_from_github(url):
    """
    Reads a CSV file from a given GitHub URL and returns it as a pandas DataFrame.
    input:
        url = URL of the CSV file on GitHub
    output:
        df = DataFrame containing the CSV data
    """
    response = requests.get(url)
    if response.status_code == 200:
        csv_content = response.content.decode('utf-8')
        return pd.read_csv(StringIO(csv_content))
    else:
        print(f"Failed to fetch file. Status code: {response.status_code}, Message: {response.text}")
        return pd.DataFrame()
        
def loading_surveillance_eval(start_date, end_date, path_surv):
    """
    This function loads the surveillance data from a GitHub repository.
    input:
        start_date = start date for the data
        github_repo = GitHub repository name
        github_directory = directory in the GitHub repository
        surveillance_file = name of the surveillance file
        state = state to filter the data
        horizon_to_start = horizon to start the data
    output:
        df_surv_date_US = dataframe with the filtered surveillance data
    """
    df_surv = pd.read_csv(path_surv)

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    df_surv['date'] = pd.to_datetime(df_surv['date'])
    df_surv = df_surv[(df_surv['target'] == 'inc hosp') & (df_surv['age_group'] == '65-130')]
    # filter the dataframe to only include rows where the location is 'US' and the date is between start_date and ref_date
    df_surv_date_US = df_surv[(df_surv.location == 'US') & 
                            (df_surv.date >= start_date) & 
                            (df_surv.date <= end_date)]
    df_surv_date_US = df_surv_date_US.rename(columns={"observation": "hospitalizations"})
    df_surv_date_US = df_surv_date_US.sort_values(by='date')
    df_surv_date_US['horizon'] = np.arange(1, len(df_surv_date_US)+1)
    return df_surv_date_US

In [ ]:
state = 'US'
# Loading surveillance data for evaluation: last surveillance file
start_date = datetime(2024, 11, 23)  # Adjusted to the latest date in the surveillance data
end_date = datetime(2025, 5, 17)
path_surv =  "../../../../input_data/target-data-age-stratification.csv"
df_surv = loading_surveillance_eval(start_date, end_date, path_surv)
df_surv

,date,location,hospitalizations,age_group,target,horizon
22725,2024-11-23,US,1572.0,65-130,inc hosp,1
22730,2024-11-30,US,2077.0,65-130,inc hosp,2
22735,2024-12-07,US,3122.0,65-130,inc hosp,3
22740,2024-12-14,US,4529.0,65-130,inc hosp,4
22745,2024-12-21,US,7556.0,65-130,inc hosp,5
22750,2024-12-28,US,14858.0,65-130,inc hosp,6
22755,2025-01-04,US,21984.0,65-130,inc hosp,7
22760,2025-01-11,US,17210.0,65-130,inc hosp,8
22765,2025-01-18,US,17547.0,65-130,inc hosp,9
22770,2025-01-25,US,21440.0,65-130,inc hosp,10


In [42]:
df_surv.tail(10)

,date,location,hospitalizations,age_group,target,horizon
22805,2025-03-15,US,10519.0,65-130,inc hosp,17
22810,2025-03-22,US,7858.0,65-130,inc hosp,18
22815,2025-03-29,US,5265.0,65-130,inc hosp,19
22820,2025-04-05,US,3561.0,65-130,inc hosp,20
22825,2025-04-12,US,2457.0,65-130,inc hosp,21
22830,2025-04-19,US,1853.0,65-130,inc hosp,22
22835,2025-04-26,US,1527.0,65-130,inc hosp,23
22840,2025-05-03,US,1180.0,65-130,inc hosp,24
22845,2025-05-10,US,977.0,65-130,inc hosp,25
22850,2025-05-17,US,920.0,65-130,inc hosp,26


In [ ]:
loss_function = 'wmape'
season = "2024-2025"
df_original_ensemble = pd.read_csv(f'../../output_data/original_ensembles_agestrata/Ensemble_Ens2_USnational.csv', index_col = 0)
df_original_ensemble

,quantile,value,horizon
1,0.010,20.030710,1
2,0.025,36.161075,1
3,0.050,42.876155,1
4,0.100,48.595233,1
5,0.150,52.356198,1
...,...,...,...
709,0.850,477.900000,31
710,0.900,507.800000,31
711,0.950,549.400000,31
712,0.975,587.710000,31


In [44]:
def import_ensemble_original(df, df_surv, date_ref):
    df_surv['date'] = pd.to_datetime(df_surv['date'])
    valid_horizons = df_surv['horizon'].unique()
    df = df[df['horizon'].isin(valid_horizons)]
    # Map each horizon to its corresponding date
    horizon_to_date = df_surv.set_index('horizon')['date'].to_dict()
    df['date'] = df['horizon'].map(horizon_to_date)
    # Filter to only include dates after or equal to date_only
    date_ref = pd.to_datetime(date_ref)
    df = df[df['date'] >= date_ref]
    # Merge in hospitalization targets
    df = df.merge(df_surv[['date', 'hospitalizations']].drop_duplicates(), on='date', how='left')
    return df

def import_ensemble2(file_path, surv_lookup, horizon_to_date):
    df = pd.read_csv(file_path, index_col=0)
    df['horizon'] = df['horizon'].astype(int)
    df['date'] = df['horizon'].map(horizon_to_date)
    df['date'] = pd.to_datetime(df['date'])
    return df.merge(surv_lookup, on='date', how='left')   

In [45]:
def get_perc_error(actual, sim, last=4): 
    return 100 * np.mean(np.abs(actual[-last:] - sim[-last:]) / actual[-last:])

def get_wmape(actual, sim) -> float:
    return np.sum(np.abs(actual - sim)) / np.sum(np.abs(actual))

def diff(a, b, norm=False): 
    if norm:
        if a != 0:
            return (a - b) / a
        else: 
            return 0
    else:
        return (a - b)

def interval_score(y, u, l, alpha, norm=False): 
    return -diff(l, u, norm=norm) + 2 / alpha * -diff(y, l, norm=norm) * (y < l) + 2 / alpha * diff(y, u, norm=norm) * (y > u)
    

def weighted_interval_score(y, u_s, l_s, m, alpha_ks, w0=1./2., norm=False):
    K = len(alpha_ks)
    wks = np.array(alpha_ks) / 2.
    return 1. / (K + 1./2.) * (w0 * np.abs(diff(y, m, norm=norm)) + np.dot(wks, [interval_score(y, u, l, a_k, norm=norm) for u, l, a_k in zip(u_s, l_s, alpha_ks)]))


def get_upper_bound(sim_stats, alpha, idx): 
    # get upper bound levels
    q2 = 1.0 - alpha / 2
    return sim_stats.loc[sim_stats["quantile"] == q2]["value"].values[idx]

def get_lower_bound(sim_stats, alpha, idx): 
    # get lower bound levels
    q1 = alpha / 2.
    return sim_stats.loc[sim_stats["quantile"] == q1]["value"].values[idx]


def get_aggregate_wis(realdata, 
                      sim_stats, 
                      alphas=[0.02, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90], 
                      aggr_fun=None, 
                      norm=False):
    wis = []
    for i in range(len(realdata)):
        wis.append(weighted_interval_score(
                            y=realdata[i], 
                            u_s=np.array([get_upper_bound(sim_stats, alpha=alpha, idx=i) for alpha in alphas]), 
                            l_s=np.array([get_lower_bound(sim_stats, alpha=alpha, idx=i) for alpha in alphas]), 
                            m=sim_stats.loc[sim_stats["quantile"] == 0.5]["value"].values[i],
                            alpha_ks=alphas, 
                            norm=norm))
    if aggr_fun != None:
        return aggr_fun(wis)
    else:
        return wis
        
def compute_dict_WIS_AE(df, alphas):
    if 'quantiles' in df.columns:
        df = df.rename(columns={"quantiles": "quantile"})
    realdata = df[df['quantile'] == 0.5]['hospitalizations'].values
    sim_stats = df.drop(columns=['date', 'hospitalizations', 'horizon'])
    wmape = get_wmape(realdata, sim_stats.loc[sim_stats["quantile"] == 0.5]["value"].values)
    wis_list = get_aggregate_wis(realdata, 
                      sim_stats, 
                      alphas=alphas, 
                      aggr_fun=None, 
                      norm=False)
    wis_mean = np.mean(wis_list)
    return wis_list, wis_mean, wmape

In [ ]:
adaptive_ensemble2_path = "../../output_data/adaptive_ensemble2_agestrata"
horizon_to_date = df_surv.set_index('horizon')['date'].to_dict()
surv_lookup = df_surv[['date', 'hospitalizations']].drop_duplicates()
k_values = [0.05, 0.15, 0.25, 0.50, 0.75]
alphas=[0.02, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]


dict_k_wis_rounds = {}
dict_k_mae_rounds = {}
for k in k_values:
    dict_wis_rounds = {}
    dict_mae_rounds = {}
    unique_dates = df_surv['date'].sort_values().unique()[:-2]
    for date in unique_dates:
        # Adaptive Ensemble2
        print(date)
        full_path_adaptive_ensemble2 = f"{adaptive_ensemble2_path}/{date.strftime('%Y-%m-%d')}_{k}_{loss_function}.csv"
        print(full_path_adaptive_ensemble2)
        df_adaptive_ensemble2 = import_ensemble2(full_path_adaptive_ensemble2, surv_lookup, horizon_to_date)
        print(df_adaptive_ensemble2)
        wis_list_adens2, wis_mean_adens2, wmape_adens2 = compute_dict_WIS_AE(df_adaptive_ensemble2, alphas)
        
        # Original ensemble
        df_original_ensemble2 = import_ensemble_original(df_original_ensemble, df_surv, date)
        wis_list_original_ens2, wis_mean_original_ens2, wmape_original_ens2 = compute_dict_WIS_AE(df_original_ensemble2, alphas)
        # Store WIS and AE results
        dict_wis_rounds[date] = [wis_mean_adens2, wis_mean_original_ens2]
        dict_mae_rounds[date] = [wmape_adens2, wmape_original_ens2]
    k_perc = int(k * 100)
    dict_k_wis_rounds[k_perc] = dict_wis_rounds
    dict_k_mae_rounds[k_perc] = dict_mae_rounds

    

2024-11-23 00:00:00
../output_data/review_adaptive_ensemble2_agestrata/2024-11-23_0.05_wmape.csv
     quantile    value  horizon       date  hospitalizations
0       0.010  136.230        3 2024-12-07            3122.0
1       0.025  147.900        3 2024-12-07            3122.0
2       0.050  151.000        3 2024-12-07            3122.0
3       0.100  160.600        3 2024-12-07            3122.0
4       0.150  167.400        3 2024-12-07            3122.0
..        ...      ...      ...        ...               ...
547     0.850  558.600       26 2025-05-17             920.0
548     0.900  596.980       26 2025-05-17             920.0
549     0.950  657.580       26 2025-05-17             920.0
550     0.975  712.680       26 2025-05-17             920.0
551     0.990  743.476       26 2025-05-17             920.0

[552 rows x 5 columns]
2024-11-30 00:00:00
../output_data/review_adaptive_ensemble2_agestrata/2024-11-30_0.05_wmape.csv
     quantile    value  horizon       date  hospit

In [49]:
dict_k_wis_rounds

{5: {Timestamp('2024-11-23 00:00:00'): [7568.682239550814, 4768.647961047382],
  Timestamp('2024-11-30 00:00:00'): [7862.203933898624, 4941.808243615357],
  Timestamp('2024-12-07 00:00:00'): [8187.080566053404, 5125.126754179272],
  Timestamp('2024-12-14 00:00:00'): [8321.071203216265, 5311.092101823443],
  Timestamp('2024-12-21 00:00:00'): [7941.546491004112, 5495.633985718677],
  Timestamp('2024-12-28 00:00:00'): [7308.34519279938, 5652.07991668768],
  Timestamp('2025-01-04 00:00:00'): [6294.093737602216, 5672.449646767913],
  Timestamp('2025-01-11 00:00:00'): [4966.813811841505, 5498.506590581535],
  Timestamp('2025-01-18 00:00:00'): [5007.429738877248, 5485.334959436747],
  Timestamp('2025-01-25 00:00:00'): [4351.807519987544, 5453.397724080024],
  Timestamp('2025-02-01 00:00:00'): [3446.9214677310333, 5248.431186594242],
  Timestamp('2025-02-08 00:00:00'): [2529.8930620926044, 4695.813263385327],
  Timestamp('2025-02-15 00:00:00'): [1785.3539232503508, 3928.4593676640425],
  Times

In [ ]:
# Create df_wis dataframe to store WIS results
rows = []

for k, group_data in dict_k_wis_rounds.items():
    for timestamp, values in group_data.items():
        rows.append({
            'k_perc': k,
            'week': timestamp,
            'wis_adaptive_ensemble2': values[0],
            'wis_original_ensemble2': values[1],
            'wis_rel_original2': values[0] / values[1],
        })
df_wis = pd.DataFrame(rows)
df_wis
df_wis.to_csv(f"../../output_data/performance_adaptive_agestrata/wis_performance_{loss_function}.csv", index=False)

In [ ]:
# Create df_mae dataframe to store MAE results
rows = []

for k, group_data in dict_k_mae_rounds.items():
    for timestamp, values in group_data.items():
        rows.append({
            'k_perc': k,
            'week': timestamp,
            'mae_adaptive_ensemble2': values[0],
            'mae_original_ensemble2': values[1],
            'mae_rel_original2': values[0] / values[1]
        })
df_mae = pd.DataFrame(rows)
df_mae.to_csv(f"../../output_data/performance_adaptive_agestrata/mae_performance_{loss_function}.csv", index=False)
df_mae

,k_perc,week,mae_adaptive_ensemble2,mae_original_ensemble2,mae_rel_original2
0,5,2024-11-23,0.893590,0.766121,1.166382
1,5,2024-11-30,0.801926,0.767795,1.044454
2,5,2024-12-07,0.804809,0.770324,1.044768
3,5,2024-12-14,0.799313,0.773769,1.033012
4,5,2024-12-21,0.788034,0.778749,1.011923
...,...,...,...,...,...
115,75,2025-04-05,0.926002,0.884763,1.046610
116,75,2025-04-12,0.944987,0.913165,1.034848
117,75,2025-04-19,0.956573,0.933324,1.024909
118,75,2025-04-26,0.964331,0.947473,1.017793
